In [ ]:
from huggingface_hub import login
login("hf_XXXXXXXXX") # 我的TOKEN

In [ ]:
import os
import numpy as np
import torch
from tqdm import tqdm
from dataclasses import dataclass
from typing import List, Optional, Dict, Tuple

from transformers import AutoTokenizer, AutoModelForCausalLM

# -----------------------------------------------
# 確保拿得到 attn_weights
# -----------------------------------------------
os.environ["PYTORCH_USE_SDPA"] = "0"
os.environ["FLASH_ATTENTION"] = "0"


# =============== Data Class for Output ===============
@dataclass
class HeadAttentionMetrics:
    layer: int
    head: int
    total_positions: int
    self_copy_ratio: float
    prev_copy_ratio: float
    other_ratio: float
    mean_entropy: float
    mean_peak_ratio: float
    copy_score: float


# =====================================================
#     ATTENTION-BASED COPY HEAD FINDER
# =====================================================
class CopyHeadFinderAttention:

    def __init__(self, model_name: str, hf_token: Optional[str], device="cuda"):

        # ---- Load tokenizer ----
        print(f"[Init] Loading tokenizer: {model_name}")
        tok_kwargs = {}
        if hf_token:
            tok_kwargs["token"] = hf_token

        self.tokenizer = AutoTokenizer.from_pretrained(model_name, **tok_kwargs)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        # ---- Load model ----
        print(f"[Init] Loading model (eager attention)...")

        model_kwargs = {
            "torch_dtype": torch.float16 if device == "cuda" else torch.float32,
            "attn_implementation": "eager",
        }
        if hf_token:
            model_kwargs["token"] = hf_token
        if device == "cuda":
            model_kwargs["device_map"] = "cuda"

        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            **model_kwargs
        )
        self.model.eval()

        self.device = device
        self.config = self.model.config

        self.num_layers = self.config.num_hidden_layers
        self.num_heads = self.config.num_attention_heads
        self.head_dim = self.config.hidden_size // self.num_heads

        print(
            f"[Init] Layers={self.num_layers}, "
            f"Heads={self.num_heads}, HeadDim={self.head_dim}, device={self.device}"
        )

    # =========================================================
    # 用 attention 找 copy heads
    # =========================================================
    def find_copy_heads(
        self,
        texts: List[str],
        top_k: Optional[int] = None,
        max_length: Optional[int] = None,
    ) -> List[HeadAttentionMetrics]:

        if top_k is None:
            top_k = self.num_layers   # 32 層 就是 32 個 best heads

        # ---- Tokenization ----
        encoded_texts = []
        for t in texts:
            enc = self.tokenizer(t, return_tensors="pt", truncation=True, max_length=max_length)
            encoded_texts.append(enc)

        # ---- 用來存全部 head 的統計 ----
        all_head_stats: Dict[Tuple[int, int], Dict] = {}

        # =====================================================
        #  逐層 Hook，計算 attention 行為
        # =====================================================
        with torch.no_grad():
            for layer_idx in range(self.num_layers):

                layer = self.model.model.layers[layer_idx].self_attn

                # 每層初始化 counters
                head_counters = {
                    h: {
                        "same": 0,
                        "prev": 0,
                        "other": 0,
                        "entropy_sum": 0.0,
                        "peak_sum": 0.0,
                        "pos_count": 0,
                    }
                    for h in range(self.num_heads)
                }

                # --------------------- hook function ---------------------
                def hook_fn(module, input, output):
                    attn_output, attn_weights = output[:2]
                    if attn_weights is None:
                        return output

                    w = attn_weights 
                    bsz, H, T, S = w.shape

                    eps = 1e-12

                    # ===== entropy =====
                    w_safe = w + eps
                    entropy = -(w_safe * torch.log(w_safe)).sum(dim=-1)
                    entropy = torch.nan_to_num(entropy, nan=0.0, posinf=0.0, neginf=0.0)

                    # ===== peak ratio =====
                    max_vals = w.max(dim=-1).values
                    sum_vals = torch.clamp(w.sum(dim=-1), min=1e-12)
                    peak = max_vals / sum_vals

                    # ===== argmax source =====
                    src_argmax = w.argmax(dim=-1)

                    # ===== accumulate =====
                    for b in range(bsz):
                        for h in range(self.num_heads):
                            ent_ht = entropy[b, h]
                            peak_ht = peak[b, h]
                            src_ht = src_argmax[b, h]

                            for tgt in range(T):
                                src = src_ht[tgt].item()

                                head_counters[h]["entropy_sum"] += float(ent_ht[tgt].item())
                                head_counters[h]["peak_sum"] += float(peak_ht[tgt].item())
                                head_counters[h]["pos_count"] += 1

                                if src == tgt:
                                    head_counters[h]["same"] += 1
                                elif src == tgt - 1:
                                    head_counters[h]["prev"] += 1
                                else:
                                    head_counters[h]["other"] += 1

                    return output

                # ---- Register hook ----
                handle = layer.register_forward_hook(hook_fn)

                # ---- Run all texts ----
                for enc in tqdm(encoded_texts, desc=f"[Layer {layer_idx}] scanning"):
                    inputs = {k: v.to(self.device) for k, v in enc.items()}
                    _ = self.model(**inputs, output_attentions=True)

                # ---- Remove hook ----
                handle.remove()

                # ---- Save stats for this layer ----
                for h in range(self.num_heads):
                    c = head_counters[h]
                    total = c["same"] + c["prev"] + c["other"]

                    if total == 0:
                        continue

                    s = {
                        "layer": layer_idx,
                        "head": h,
                        "total_positions": total,
                        "self_copy_ratio": c["same"] / total,
                        "prev_copy_ratio": c["prev"] / total,
                        "other_ratio": c["other"] / total,
                        "mean_entropy": c["entropy_sum"] / c["pos_count"],
                        "mean_peak_ratio": c["peak_sum"] / c["pos_count"],
                    }
                    all_head_stats[(layer_idx, h)] = s

        # =====================================================
        #   GLOBAL NORMALIZATION + COPY SCORE
        # =====================================================
        entropies = [s["mean_entropy"] for s in all_head_stats.values()]
        peaks = [s["mean_peak_ratio"] for s in all_head_stats.values()]

        ent_min, ent_max = min(entropies), max(entropies)
        peak_min, peak_max = min(peaks), max(peaks)

        def norm(x, x_min, x_max):
            if abs(x_max - x_min) < 1e-12:
                return 0.5
            return (x - x_min) / (x_max - x_min)

        head_metrics: List[HeadAttentionMetrics] = []

        for key, s in all_head_stats.items():
            layer_idx, head_idx = key

            self_r = s["self_copy_ratio"]
            prev_r = s["prev_copy_ratio"]
            ent = s["mean_entropy"]
            peak = s["mean_peak_ratio"]

            ent_norm = norm(ent, ent_min, ent_max)
            peak_norm = norm(peak, peak_min, peak_max)

            # ===== copy score =====
            copy_score = (
                0.45 * prev_r + 
                0.35 * self_r +
                0.1 * peak_norm +
                0.1 * (1.0 - ent_norm)
            )
            if np.isnan(copy_score):
                copy_score = 0.0

            head_metrics.append(
                HeadAttentionMetrics(
                    layer=layer_idx,
                    head=head_idx,
                    total_positions=s["total_positions"],
                    self_copy_ratio=self_r,
                    prev_copy_ratio=prev_r,
                    other_ratio=s["other_ratio"],
                    mean_entropy=ent,
                    mean_peak_ratio=peak,
                    copy_score=copy_score,
                )
            )

        # ===== sort + return top_k =====
        sorted_heads = sorted(head_metrics, key=lambda x: x.copy_score, reverse=True)
        return sorted_heads[:top_k]

In [ ]:
# ==============================
#            Usage
# ==============================

from transformers import AutoConfig, AutoModelForCausalLM

if __name__ == "__main__":

    MODEL_KEY = "Llama-1B"

    if MODEL_KEY == "Llama-1B":
        MODEL = "meta-llama/Llama-3.2-1B-Instruct"
    elif MODEL_KEY == "Llama-3B":
        MODEL = "meta-llama/Llama-3.2-3B-Instruct"
    
    TOKEN = "hf_XXXXXXX" # 請換成自己的

    config = AutoConfig.from_pretrained(MODEL, token=TOKEN)

    print("Transformer 層數:", config.num_hidden_layers)
    print("Hidden size:", config.hidden_size)
    print("Attention heads:", config.num_attention_heads)

    TOP_K = config.num_hidden_layers


    finder = CopyHeadFinderAttention(
        model_name=MODEL,
        hf_token=TOKEN,
        device="cuda"
    )

    TEST_TEXTS = [
        "A B C D E F G H I J K L M N O P Q R S",
        "1 2 3 4 5 6 7 8 9 10 11 12 13 14",
        "hello world " * 15,
    ]

    top_heads = finder.find_copy_heads(TEST_TEXTS, top_k=TOP_K, max_length=5192)

    print("\n=== TOP COPY HEADS (Attention-based) ===")
    for h in top_heads:
        print(
            f"L{h.layer:2d} H{h.head:2d} | score={h.copy_score:.4f} | "
            f"prev={h.prev_copy_ratio:.3f} self={h.self_copy_ratio:.3f} "
            f"entropy={h.mean_entropy:.3f} peak={h.mean_peak_ratio:.3f}"
        )


    top_pairs = [[h.layer, h.head] for h in top_heads]

    print("\n=== TOP COPY HEADS (as index pairs) ===")
    print(top_pairs)

In [ ]:
import json

with open(f"top_copy_head_pairs_for_{MODEL_KEY}.json", "w", encoding="utf-8") as f:
    json.dump(top_pairs, f, indent=4, ensure_ascii=False)

print(f"已成功輸出：top_copy_head_pairs_for_{MODEL_KEY}.json")